# Fuga da Masmorra de Pântano

**Estudantes:** `NOME DO ESTUDANTE 1` e `NOME DO ESTUDANTE 2`.

- Estudante 1: `descrever contribuição`.
- Estudante 2: `descrever contribuição`.

Um mago aprendiz explora a Cripta do Guardião para recuperar uma chave e abrir a porta selada. Desenhamos dois cenários do mesmo jogo: a **ala externa**, com quatro salas, e a **fortaleza inundada**, com dezesseis salas, passagens estreitas, pântanos e slimes. O trabalho compara UCS e A* ao buscar uma rota de fuga em cada cenário.

## Modelagem do problema

- **Espaço de estados:** cada estado é a tupla `(linha, coluna, possui_chave)`, que representa a posição do mago e se ele já obteve a chave.
- **Estado inicial:** posição marcada com `S`, com `possui_chave = False`.
- **Condição objetivo:** o mago ocupa a célula `E` e `possui_chave = True`.
- **Ações:** mover uma célula para cima, baixo, esquerda ou direita. Não há movimentos diagonais.
- **Restrições:** paredes de pedra (`#`) são intransponíveis; a porta (`E`) permanece bloqueada até o mago pegar a chave (`K`).
- **Custo de uma ação:** é o custo do terreno da célula de destino. Piso de pedra (`.`) custa 1, pântano (`,`) custa 3 e slime (`L`) custa 6.

As paredes separam salas e corredores. Os dois mapas são definidos e exibidos a seguir; a ala externa e a fortaleza são comparadas ao final do notebook. Uma rota com menos movimentos pode custar mais se atravessar pântanos ou slimes.

In [121]:
# Dois mapas desenhados manualmente. S=início, K=chave, E=saída.
MAPA_FACIL = [
    "#################",
    "#.......#.......#",
    "#.S.....#.......#",
    "#..##...#..LL#..#",
    "#..#.......LL#..#",
    "#.......#.......#",
    "#.......#.......#",
    "#.......#.......#",
    "############.####",
    "###....##.......#",
    "#.,,,,,.#.......#",
    "#.,,.##.#....,,.#",
    "#.,,..#.L..#L,,.#",
    "#.,,K...#...LE,.#",
    "#.,,,..##,,...L.#",
    "##########,,....#",
    "#################",
]

MAPA_DIFICIL = [
    "#########################################",
    "#.........#.........#.........#.........#",
    "#.S.......#......#..#.........#.......E.#",
    "#..##.....#..,.,,,,.#.........#..,,,.#..#",
    "#..#......,....,,,,.#...LLL...#..,,,.#..#",
    "#.........#....,,,,.#...LLL...,..,,,....#",
    "#.........#...,,....L...LLL...#..,,,....#",
    "#......##.#.##......#.........#.#.......#",
    "#.........#..,,,##..#.........#.........#",
    "#.........#,,L,,,...#.........#.........#",
    "##############L#.########.########.######",
    "#.........#...L.....#.........#.........#",
    "#..,,,.#..#.#.L..#..#.......#.#..LLL.#..#",
    "#..,,,.#..#..LLL....#.......#.#..LLL.#..#",
    "#..,,,....#..LLL.......,,,,...#..LLL....#",
    "#..,,,....#..LLL....#..,,,,...#..LLL....#",
    "#..,,,....#...L.....#..,,,,.............#",
    "#..##.......#.L..#..#.#,,,,...#.#.......#",
    "#.........#...L...,.#.........#.........#",
    "#.........#...L.....#.........#.........#",
    "####,##########L###########.#######.#####",
    "####................#LLLLLLLLL#.........#",
    "#..######.#.#..,.#..#.......#.#......#..#",
    "#..LLL..###.....,...#...LLL.#.#......#..#",
    "#..LLL....#..,,,.,,.#...LLL......,,,....#",
    "#..LLL....#..,,,..,.#...LLL...#..,,,....#",
    "#..LLL.......,,,..,.#...LLL...#..,,,....#",
    "#......#..#.#,,,.#....#.......#.#,,,....#",
    "#.........#.........#.........#.........#",
    "#.........#.........#.........#.........#",
    "#.########################.#######.######",
    "#.#.......#LLLLLLLLL#....LLLL.#.........#",
    "#.#....#..#.#....#..#.......#.#......#..#",
    "#.#....#..#..LLL....#.......#.#..LLLL#..#",
    "#.#..........LLL....#..,,,....#..LLLL...#",
    "#.#.......#..LLL....#..,,,.......LLLL...#",
    "#.LLLLL...#..LLL.......,,,....#.........#",
    "#..K.#....#......#..#.#.......#.#.......#",
    "#.#....#..#.........#.........#.........#",
    "#..#...####.........#.........#.........#",
    "#########################################",
]

CENARIOS = {
    "Fácil — ala externa": MAPA_FACIL,
    "Difícil — fortaleza inundada": MAPA_DIFICIL,
}

# A demonstração passo a passo das buscas usa a fortaleza.
# A comparação final executa todos os métodos nos dois cenários.
DUNGEON = list(MAPA_DIFICIL)

CUSTOS = {
    ".": 1,  # piso de pedra
    ",": 3,  # pântano
    "L": 6,  # slime
    "S": 1,
    "K": 1,
    "E": 1,
}

ACOES = {
    "cima": (-1, 0),
    "baixo": (1, 0),
    "esquerda": (0, -1),
    "direita": (0, 1),
}

for nome, mapa in CENARIOS.items():
    assert len({len(linha) for linha in mapa}) == 1, f"{nome}: linhas de tamanhos diferentes."
    assert all(sum(linha.count(simbolo) for linha in mapa) == 1
               for simbolo in ("S", "K", "E")), f"{nome}: S, K e E devem ser únicos."
    print(f"{nome}: {len(mapa)} × {len(mapa[0])} células")

N_LINHAS, N_COLUNAS = len(DUNGEON), len(DUNGEON[0])

Fácil — ala externa: 17 × 17 células
Difícil — fortaleza inundada: 41 × 41 células


In [122]:
PAREDE = "#"
CHAVE = "K"
SAIDA = "E"

def encontrar_posicao_no_mapa(simbolo, mapa=None):
    """Encontra um símbolo no mapa informado ou no cenário ativo."""
    mapa_consultado = DUNGEON if mapa is None else mapa
    for indice_linha, linha_do_mapa in enumerate(mapa_consultado):
        for indice_coluna, celula in enumerate(linha_do_mapa):
            if celula == simbolo:
                return (indice_linha, indice_coluna)

    raise ValueError(f"Símbolo {simbolo!r} não encontrado na dungeon.")


POSICAO_INICIAL = encontrar_posicao_no_mapa("S")
POSICAO_CHAVE = encontrar_posicao_no_mapa(CHAVE)
POSICAO_SAIDA = encontrar_posicao_no_mapa(SAIDA)
ESTADO_INICIAL = (*POSICAO_INICIAL, False)


def posicao_esta_dentro_do_mapa(linha, coluna):
    """Verifica se uma coordenada está dentro dos limites da dungeon."""
    linha_eh_valida = 0 <= linha < N_LINHAS
    coluna_eh_valida = 0 <= coluna < N_COLUNAS
    return linha_eh_valida and coluna_eh_valida


def estado_eh_objetivo(estado):
    """Retorna True somente se o mago alcançar a saída portando a chave."""
    linha_atual, coluna_atual, mago_possui_chave = estado
    mago_esta_na_saida = (linha_atual, coluna_atual) == POSICAO_SAIDA

    return mago_esta_na_saida and mago_possui_chave


def gerar_sucessores(estado_atual):
    """Retorna uma lista de (ação, próximo_estado, custo) para movimentos permitidos."""
    linha_atual, coluna_atual, mago_possui_chave = estado_atual
    movimentos_validos = []

    for nome_da_acao, (variacao_linha, variacao_coluna) in ACOES.items():
        nova_linha = linha_atual + variacao_linha
        nova_coluna = coluna_atual + variacao_coluna

        if not posicao_esta_dentro_do_mapa(nova_linha, nova_coluna):
            continue

        celula_de_destino = DUNGEON[nova_linha][nova_coluna]
        destino_eh_parede = celula_de_destino == PAREDE
        porta_esta_trancada = celula_de_destino == SAIDA and not mago_possui_chave

        if destino_eh_parede or porta_esta_trancada:
            continue

        mago_obtem_chave_neste_movimento = celula_de_destino == CHAVE
        mago_tera_chave = mago_possui_chave or mago_obtem_chave_neste_movimento
        proximo_estado = (nova_linha, nova_coluna, mago_tera_chave)
        custo_do_movimento = CUSTOS[celula_de_destino]

        movimento = (nome_da_acao, proximo_estado, custo_do_movimento)
        movimentos_validos.append(movimento)

    return movimentos_validos


for nome, mapa in CENARIOS.items():
    print(f"{nome}: início {encontrar_posicao_no_mapa('S', mapa)}, "
          f"chave {encontrar_posicao_no_mapa(CHAVE, mapa)}, "
          f"saída {encontrar_posicao_no_mapa(SAIDA, mapa)}")
print("Movimentos iniciais na fortaleza:", gerar_sucessores(ESTADO_INICIAL))

Fácil — ala externa: início (2, 2), chave (13, 4), saída (13, 13)
Difícil — fortaleza inundada: início (2, 2), chave (37, 3), saída (2, 38)
Movimentos iniciais na fortaleza: [('cima', (1, 2, False), 1), ('baixo', (3, 2, False), 1), ('esquerda', (2, 1, False), 1), ('direita', (2, 3, False), 1)]


In [123]:
SIMBOLOS_VISUAIS = {
    "#": "🧱",  # parede
    ".": "▫️",  # piso de pedra
    ",": "🟫",  # pântano
    "L": "🟢",  # slime
    "S": "🧙",  # mago no início
    "K": "🔑",  # chave do Guardião
    "E": "🚪",  # porta selada
}


def imprimir_mapa(caminho_de_estados=None, mapa=None):
    """Mostra um dos mapas e, se informado, destaca um caminho com pegadas."""
    mapa_exibido = DUNGEON if mapa is None else mapa
    posicoes_do_caminho = set()

    if caminho_de_estados is not None:
        posicoes_do_caminho = {
            (linha, coluna)
            for linha, coluna, possui_chave in caminho_de_estados
        }

    print("Legenda: 🧙 início | 🔑 chave | 🚪 saída | 👣 caminho | 🧱 parede | ▫️ pedra | 🟫 pântano | 🟢 slime")
    print()

    for indice_linha, linha_do_mapa in enumerate(mapa_exibido):
        linha_visual = []

        for indice_coluna, celula in enumerate(linha_do_mapa):
            posicao_atual = (indice_linha, indice_coluna)
            celula_faz_parte_do_caminho = posicao_atual in posicoes_do_caminho
            celula_eh_elemento_especial = celula in {"S", "K", "E"}

            if celula_faz_parte_do_caminho and not celula_eh_elemento_especial:
                linha_visual.append("👣")
            else:
                linha_visual.append(SIMBOLOS_VISUAIS[celula])

        print("".join(linha_visual))


for nome, mapa in CENARIOS.items():
    print(f"\n{nome} — {len(mapa)} × {len(mapa[0])}")
    imprimir_mapa(mapa=mapa)



Fácil — ala externa — 17 × 17
Legenda: 🧙 início | 🔑 chave | 🚪 saída | 👣 caminho | 🧱 parede | ▫️ pedra | 🟫 pântano | 🟢 slime

🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱
🧱▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️🧱
🧱▫️🧙▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️🧱
🧱▫️▫️🧱🧱▫️▫️▫️🧱▫️▫️🟢🟢🧱▫️▫️🧱
🧱▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️🟢🟢🧱▫️▫️🧱
🧱▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️🧱
🧱▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️🧱
🧱▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️🧱
🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱▫️🧱🧱🧱🧱
🧱🧱🧱▫️▫️▫️▫️🧱🧱▫️▫️▫️▫️▫️▫️▫️🧱
🧱▫️🟫🟫🟫🟫🟫▫️🧱▫️▫️▫️▫️▫️▫️▫️🧱
🧱▫️🟫🟫▫️🧱🧱▫️🧱▫️▫️▫️▫️🟫🟫▫️🧱
🧱▫️🟫🟫▫️▫️🧱▫️🟢▫️▫️🧱🟢🟫🟫▫️🧱
🧱▫️🟫🟫🔑▫️▫️▫️🧱▫️▫️▫️🟢🚪🟫▫️🧱
🧱▫️🟫🟫🟫▫️▫️🧱🧱🟫🟫▫️▫️▫️🟢▫️🧱
🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🟫🟫▫️▫️▫️▫️🧱
🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱

Difícil — fortaleza inundada — 41 × 41
Legenda: 🧙 início | 🔑 chave | 🚪 saída | 👣 caminho | 🧱 parede | ▫️ pedra | 🟫 pântano | 🟢 slime

🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱
🧱▫️▫️▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️▫️▫️🧱
🧱▫️🧙▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️🧱▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️🚪▫️🧱
🧱▫️▫️🧱🧱▫️▫️▫️▫️▫️🧱▫️▫️🟫▫️🟫🟫🟫🟫▫️🧱▫️▫️▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️🟫🟫🟫▫️🧱▫️▫️🧱
🧱▫️▫️🧱▫️▫️▫️▫️▫️▫️🟫▫️▫️▫️▫️

In [124]:
# Testes das regras do problema

def testar_estado_inicial_nao_e_objetivo():
    """O mago não pode vencer antes de encontrar a chave e chegar à saída."""
    estado_inicial_eh_objetivo = estado_eh_objetivo(ESTADO_INICIAL)

    assert not estado_inicial_eh_objetivo, (
        "O estado inicial foi considerado objetivo, mas o mago ainda não possui a chave."
    )


def testar_condicao_de_vitoria_exige_chave():
    """A saída só é um objetivo se o mago estiver portando a chave."""
    estado_na_saida_com_chave = (*POSICAO_SAIDA, True)
    estado_na_saida_sem_chave = (*POSICAO_SAIDA, False)

    saida_com_chave_eh_objetivo = estado_eh_objetivo(estado_na_saida_com_chave)
    saida_sem_chave_nao_eh_objetivo = not estado_eh_objetivo(estado_na_saida_sem_chave)

    assert saida_com_chave_eh_objetivo, (
        "A saída com a chave deveria satisfazer a condição de vitória."
    )
    assert saida_sem_chave_nao_eh_objetivo, (
        "A saída sem a chave não pode satisfazer a condição de vitória."
    )


def testar_porta_bloqueada_sem_chave():
    """Nenhuma célula vizinha válida deve permitir entrar na saída sem a chave."""
    linha_saida, coluna_saida = POSICAO_SAIDA
    vizinhos_da_saida = [
        (linha_saida - 1, coluna_saida),  # acima
        (linha_saida + 1, coluna_saida),  # abaixo
        (linha_saida, coluna_saida - 1),  # esquerda
        (linha_saida, coluna_saida + 1),  # direita
    ]

    for linha, coluna in vizinhos_da_saida:
        posicao_eh_valida = (
            posicao_esta_dentro_do_mapa(linha, coluna)
            and DUNGEON[linha][coluna] != PAREDE
        )
        if not posicao_eh_valida:
            continue

        estado_vizinho_sem_chave = (linha, coluna, False)
        movimentos_possiveis = gerar_sucessores(estado_vizinho_sem_chave)
        porta_foi_oferecida_como_movimento = any(
            proximo_estado[:2] == POSICAO_SAIDA
            for _, proximo_estado, _ in movimentos_possiveis
        )

        assert not porta_foi_oferecida_como_movimento, (
            f"A porta foi acessível sem chave a partir de {(linha, coluna)}."
        )


testar_estado_inicial_nao_e_objetivo()
testar_condicao_de_vitoria_exige_chave()
testar_porta_bloqueada_sem_chave()

print("Todos os testes das regras foram aprovados.")

Todos os testes das regras foram aprovados.


In [125]:
# UCS — Busca de Custo Uniforme
# A fronteira é ordenada pelo custo acumulado para chegar a cada estado.

import heapq
from time import perf_counter


def busca_custo_uniforme(estado_inicial):
    """Encontra uma solução de menor custo usando UCS."""
    inicio_da_execucao = perf_counter()

    # Cada item: (custo, desempate, estado, caminho, ações).
    contador_de_desempate = 0
    fronteira = [(0, contador_de_desempate, estado_inicial, [estado_inicial], [])]

    # Guarda o menor custo conhecido para cada estado.
    menor_custo_por_estado = {estado_inicial: 0}
    total_nos_visitados = 0
    maior_tamanho_da_fronteira = 1

    while fronteira:
        custo_atual, _, estado_atual, caminho_atual, acoes_realizadas = heapq.heappop(fronteira)

        # Ignora caminhos antigos quando já existe um mais barato para o mesmo estado.
        menor_custo_conhecido = menor_custo_por_estado[estado_atual]
        if custo_atual > menor_custo_conhecido:
            continue

        total_nos_visitados += 1

        if estado_eh_objetivo(estado_atual):
            tempo_de_execucao = perf_counter() - inicio_da_execucao
            return {
                "encontrou_solucao": True,
                "caminho_estados": caminho_atual,
                "acoes": acoes_realizadas,
                "custo_total": custo_atual,
                "total_nos_visitados": total_nos_visitados,
                "tamanho_da_solucao": len(acoes_realizadas),
                "tempo_de_execucao": tempo_de_execucao,
                "maior_tamanho_da_fronteira": maior_tamanho_da_fronteira,
            }

        for acao, proximo_estado, custo_do_movimento in gerar_sucessores(estado_atual):
            novo_custo = custo_atual + custo_do_movimento
            custo_anterior = menor_custo_por_estado.get(proximo_estado, float("inf"))

            # Insere o sucessor somente se este for o melhor caminho até ele.
            if novo_custo < custo_anterior:
                menor_custo_por_estado[proximo_estado] = novo_custo
                novo_caminho = caminho_atual + [proximo_estado]
                novas_acoes = acoes_realizadas + [acao]

                contador_de_desempate += 1
                item_da_fronteira = (
                    novo_custo,
                    contador_de_desempate,
                    proximo_estado,
                    novo_caminho,
                    novas_acoes,
                )
                heapq.heappush(fronteira, item_da_fronteira)

                maior_tamanho_da_fronteira = max(
                    maior_tamanho_da_fronteira,
                    len(fronteira),
                )

    # Este retorno só ocorre caso não exista uma solução.
    tempo_de_execucao = perf_counter() - inicio_da_execucao
    return {
        "encontrou_solucao": False,
        "caminho_estados": [],
        "acoes": [],
        "custo_total": None,
        "total_nos_visitados": total_nos_visitados,
        "tamanho_da_solucao": 0,
        "tempo_de_execucao": tempo_de_execucao,
        "maior_tamanho_da_fronteira": maior_tamanho_da_fronteira,
    }


def exibir_resultado_da_busca(nome_do_algoritmo, resultado):
    """Imprime a solução e as métricas solicitadas no trabalho."""
    print(f"--- {nome_do_algoritmo} ---")

    if not resultado["encontrou_solucao"]:
        print("Nenhuma solução foi encontrada.")
        return

    sequencia_de_acoes = " → ".join(resultado["acoes"])
    print(f"Sequência de ações: {sequencia_de_acoes}")
    print(f"Custo total da solução: {resultado['custo_total']}")
    print(f"Total de nós visitados: {resultado['total_nos_visitados']}")
    print(f"Tamanho da solução: {resultado['tamanho_da_solucao']} movimentos")
    print(f"Tempo de execução: {resultado['tempo_de_execucao']:.6f} segundos")
    print(f"Maior tamanho da fronteira: {resultado['maior_tamanho_da_fronteira']}")
    print()
    print("Mapa com o caminho encontrado:")
    imprimir_mapa(resultado["caminho_estados"])


resultado_ucs = busca_custo_uniforme(ESTADO_INICIAL)
exibir_resultado_da_busca("UCS — Busca de Custo Uniforme", resultado_ucs)

--- UCS — Busca de Custo Uniforme ---
Sequência de ações: direita → direita → direita → baixo → baixo → direita → direita → direita → direita → direita → direita → baixo → baixo → direita → direita → direita → baixo → direita → baixo → baixo → direita → baixo → baixo → baixo → baixo → baixo → baixo → baixo → baixo → baixo → baixo → esquerda → baixo → baixo → esquerda → baixo → baixo → esquerda → esquerda → baixo → baixo → baixo → esquerda → esquerda → esquerda → baixo → baixo → baixo → esquerda → esquerda → esquerda → esquerda → esquerda → esquerda → esquerda → esquerda → baixo → baixo → baixo → baixo → baixo → baixo → baixo → baixo → direita → direita → esquerda → esquerda → cima → cima → cima → cima → cima → cima → cima → cima → cima → cima → direita → direita → direita → direita → direita → cima → direita → direita → direita → direita → direita → cima → cima → cima → cima → cima → direita → direita → direita → direita → cima → cima → cima → cima → cima → direita → cima → cima → dire

In [126]:
# A* — implementação reutilizável
# Para trocar a heurística, basta passar outra função em funcao_heuristica.


def busca_a_estrela(estado_inicial, funcao_heuristica):
    """
    Executa A* a partir de um estado inicial.

    Parâmetros:
        estado_inicial: estado no formato (linha, coluna, possui_chave).
        funcao_heuristica: função que recebe um estado e retorna h(n).
    """
    inicio_da_execucao = perf_counter()

    # A prioridade da fronteira é f(n) = g(n) + h(n).
    custo_inicial = 0
    heuristica_inicial = funcao_heuristica(estado_inicial)
    prioridade_inicial = custo_inicial + heuristica_inicial

    # Cada item: (prioridade f, desempate, custo g, estado, caminho, ações).
    contador_de_desempate = 0
    fronteira = [
        (
            prioridade_inicial,
            contador_de_desempate,
            custo_inicial,
            estado_inicial,
            [estado_inicial],
            [],
        )
    ]

    # Guarda o menor custo g(n) conhecido para chegar a cada estado.
    menor_custo_por_estado = {estado_inicial: custo_inicial}
    total_nos_visitados = 0
    maior_tamanho_da_fronteira = 1

    while fronteira:
        (
            _,
            _,
            custo_atual,
            estado_atual,
            caminho_atual,
            acoes_realizadas,
        ) = heapq.heappop(fronteira)

        # Descarta uma entrada antiga caso um caminho mais barato tenha sido encontrado.
        menor_custo_conhecido = menor_custo_por_estado[estado_atual]
        if custo_atual > menor_custo_conhecido:
            continue

        total_nos_visitados += 1

        if estado_eh_objetivo(estado_atual):
            tempo_de_execucao = perf_counter() - inicio_da_execucao
            return {
                "encontrou_solucao": True,
                "caminho_estados": caminho_atual,
                "acoes": acoes_realizadas,
                "custo_total": custo_atual,
                "total_nos_visitados": total_nos_visitados,
                "tamanho_da_solucao": len(acoes_realizadas),
                "tempo_de_execucao": tempo_de_execucao,
                "maior_tamanho_da_fronteira": maior_tamanho_da_fronteira,
            }

        for acao, proximo_estado, custo_do_movimento in gerar_sucessores(estado_atual):
            novo_custo = custo_atual + custo_do_movimento
            custo_anterior = menor_custo_por_estado.get(proximo_estado, float("inf"))

            if novo_custo < custo_anterior:
                menor_custo_por_estado[proximo_estado] = novo_custo
                nova_heuristica = funcao_heuristica(proximo_estado)
                nova_prioridade = novo_custo + nova_heuristica
                novo_caminho = caminho_atual + [proximo_estado]
                novas_acoes = acoes_realizadas + [acao]

                contador_de_desempate += 1
                item_da_fronteira = (
                    nova_prioridade,
                    contador_de_desempate,
                    novo_custo,
                    proximo_estado,
                    novo_caminho,
                    novas_acoes,
                )
                heapq.heappush(fronteira, item_da_fronteira)

                maior_tamanho_da_fronteira = max(
                    maior_tamanho_da_fronteira,
                    len(fronteira),
                )

    tempo_de_execucao = perf_counter() - inicio_da_execucao
    return {
        "encontrou_solucao": False,
        "caminho_estados": [],
        "acoes": [],
        "custo_total": None,
        "total_nos_visitados": total_nos_visitados,
        "tamanho_da_solucao": 0,
        "tempo_de_execucao": tempo_de_execucao,
        "maior_tamanho_da_fronteira": maior_tamanho_da_fronteira,
    }


# Exemplo de uso, após definir uma heurística:
# resultado_a_estrela = busca_a_estrela(ESTADO_INICIAL, minha_heuristica)
# exibir_resultado_da_busca("A*", resultado_a_estrela)

## A* com heurística admissível

A heurística usa a distância de Manhattan. Sem a chave, o mago obrigatoriamente precisa chegar à chave e, depois, à saída; por isso estimamos `distância até a chave + distância da chave até a saída`. Com a chave, estimamos somente a distância até a saída. O resultado é multiplicado pelo menor custo possível de um movimento.

A heurística é admissível porque ignora paredes, pântanos e slimes. Ignorar esses obstáculos torna o caminho estimado igual ou menor que o custo real, nunca maior.

In [127]:
CUSTO_MINIMO_DE_MOVIMENTO = min(CUSTOS.values())


def distancia_manhattan(posicao_origem, posicao_destino):
    """Calcula a distância em movimentos ortogonais entre duas posições."""
    linha_origem, coluna_origem = posicao_origem
    linha_destino, coluna_destino = posicao_destino

    diferenca_de_linhas = abs(linha_origem - linha_destino)
    diferenca_de_colunas = abs(coluna_origem - coluna_destino)
    return diferenca_de_linhas + diferenca_de_colunas


def heuristica_admissivel(estado):
    """Estima um limite inferior do custo restante até vencer a dungeon."""
    linha_atual, coluna_atual, mago_possui_chave = estado
    posicao_atual = (linha_atual, coluna_atual)

    if mago_possui_chave:
        movimentos_minimos_restantes = distancia_manhattan(
            posicao_atual,
            POSICAO_SAIDA,
        )
    else:
        distancia_ate_a_chave = distancia_manhattan(
            posicao_atual,
            POSICAO_CHAVE,
        )
        distancia_da_chave_ate_a_saida = distancia_manhattan(
            POSICAO_CHAVE,
            POSICAO_SAIDA,
        )
        movimentos_minimos_restantes = (
            distancia_ate_a_chave + distancia_da_chave_ate_a_saida
        )

    return movimentos_minimos_restantes * CUSTO_MINIMO_DE_MOVIMENTO


resultado_a_estrela_admissivel = busca_a_estrela(
    ESTADO_INICIAL,
    heuristica_admissivel,
)

# A* com heurística admissível deve manter a otimalidade da UCS.
assert resultado_a_estrela_admissivel["custo_total"] == resultado_ucs["custo_total"], (
    "A* admissível deveria encontrar o mesmo custo ótimo encontrado pela UCS."
)

exibir_resultado_da_busca(
    "A* — heurística admissível (Manhattan)",
    resultado_a_estrela_admissivel,
)

--- A* — heurística admissível (Manhattan) ---
Sequência de ações: direita → direita → direita → baixo → baixo → direita → direita → direita → direita → direita → direita → baixo → baixo → direita → direita → direita → baixo → direita → baixo → baixo → direita → baixo → baixo → baixo → baixo → baixo → baixo → baixo → baixo → baixo → baixo → esquerda → baixo → baixo → esquerda → baixo → baixo → esquerda → esquerda → baixo → baixo → baixo → esquerda → esquerda → esquerda → baixo → baixo → baixo → esquerda → esquerda → esquerda → esquerda → esquerda → esquerda → esquerda → esquerda → baixo → baixo → baixo → baixo → baixo → baixo → baixo → baixo → direita → direita → esquerda → esquerda → cima → cima → cima → cima → cima → cima → cima → cima → cima → cima → direita → direita → direita → direita → direita → cima → direita → direita → direita → direita → direita → cima → cima → cima → cima → cima → direita → direita → direita → direita → cima → cima → cima → cima → cima → direita → cima → ci

In [128]:
# A* com heurística não admissível

FATOR_DE_SUPERESTIMACAO = 2


def heuristica_nao_admissivel(estado):
    """Superestima a distância restante para priorizar caminhos aparentemente diretos."""
    estimativa_admissivel = heuristica_admissivel(estado)
    return FATOR_DE_SUPERESTIMACAO * estimativa_admissivel


# Há um contraexemplo em qualquer mapa solucionável: com a chave, a partir
# de uma célula vizinha à saída, o custo real restante é 1, mas h = 2.
vizinhos_da_saida_com_chave = [
    (POSICAO_SAIDA[0] + dl, POSICAO_SAIDA[1] + dc, True)
    for dl, dc in ACOES.values()
    if posicao_esta_dentro_do_mapa(POSICAO_SAIDA[0] + dl, POSICAO_SAIDA[1] + dc)
    and DUNGEON[POSICAO_SAIDA[0] + dl][POSICAO_SAIDA[1] + dc] != PAREDE
]
assert any(heuristica_nao_admissivel(estado) > CUSTOS[SAIDA]
           for estado in vizinhos_da_saida_com_chave)

estimativa_inicial_nao_admissivel = heuristica_nao_admissivel(ESTADO_INICIAL)
custo_otimo_encontrado_pela_ucs = resultado_ucs["custo_total"]


resultado_a_estrela_nao_admissivel = busca_a_estrela(
    ESTADO_INICIAL,
    heuristica_nao_admissivel,
)

assert resultado_a_estrela_nao_admissivel["encontrou_solucao"], (
    "A* com a heurística não admissível deveria encontrar uma solução."
)

print(f"Estimativa não admissível no início: {estimativa_inicial_nao_admissivel}")
print(f"Custo ótimo da UCS: {custo_otimo_encontrado_pela_ucs}")

exibir_resultado_da_busca(
    "A* — heurística não admissível (2 × Manhattan)",
    resultado_a_estrela_nao_admissivel,
)

Estimativa não admissível no início: 212
Custo ótimo da UCS: 162
--- A* — heurística não admissível (2 × Manhattan) ---
Sequência de ações: direita → direita → direita → baixo → baixo → direita → direita → direita → direita → direita → direita → baixo → baixo → direita → direita → direita → baixo → direita → baixo → baixo → direita → baixo → baixo → baixo → baixo → baixo → baixo → baixo → baixo → baixo → baixo → esquerda → baixo → baixo → esquerda → baixo → baixo → esquerda → esquerda → baixo → baixo → baixo → esquerda → esquerda → esquerda → baixo → baixo → baixo → esquerda → esquerda → esquerda → esquerda → esquerda → esquerda → esquerda → esquerda → baixo → baixo → baixo → baixo → baixo → baixo → baixo → baixo → direita → direita → direita → baixo → direita → direita → cima → direita → cima → cima → cima → direita → direita → direita → direita → cima → direita → direita → cima → direita → direita → direita → baixo → direita → direita → direita → baixo → baixo → baixo → direita → dir

## Alternância entre os dois cenários

Os mapas estão escritos célula por célula no início do notebook. A função abaixo apenas seleciona um deles para as buscas; ela não cria mapas. Cada cenário contém exatamente um início, uma chave e uma saída.

In [129]:
def configurar_mapa_ativo(mapa):
    """Seleciona um dos mapas manuais e atualiza as posições usadas pelas buscas."""
    global DUNGEON, N_LINHAS, N_COLUNAS
    global POSICAO_INICIAL, POSICAO_CHAVE, POSICAO_SAIDA, ESTADO_INICIAL

    if not isinstance(mapa, (list, tuple)) or not mapa or any(not isinstance(linha, str) for linha in mapa):
        raise ValueError("O mapa deve ser uma lista não vazia de strings.")
    if len(mapa) < 5 or len(mapa[0]) < 5 or any(len(linha) != len(mapa[0]) for linha in mapa):
        raise ValueError("O mapa deve ser retangular e ter ao menos 5 × 5 células.")
    if any(celula not in set(CUSTOS) | {PAREDE} for linha in mapa for celula in linha):
        raise ValueError("O mapa contém símbolos não suportados.")
    if any(sum(linha.count(simbolo) for linha in mapa) != 1 for simbolo in ("S", CHAVE, SAIDA)):
        raise ValueError("O mapa deve conter exatamente um S, um K e um E.")
    if (set(mapa[0]) != {PAREDE} or set(mapa[-1]) != {PAREDE}
            or any(linha[0] != PAREDE or linha[-1] != PAREDE for linha in mapa)):
        raise ValueError("O mapa deve ter bordas de parede.")

    DUNGEON = list(mapa)
    N_LINHAS, N_COLUNAS = len(DUNGEON), len(DUNGEON[0])
    POSICAO_INICIAL = encontrar_posicao_no_mapa("S")
    POSICAO_CHAVE = encontrar_posicao_no_mapa(CHAVE)
    POSICAO_SAIDA = encontrar_posicao_no_mapa(SAIDA)
    ESTADO_INICIAL = (*POSICAO_INICIAL, False)


## Comparação experimental: ala externa e fortaleza inundada

Os dois casos foram desenhados manualmente. A ala externa é o **caso fácil**, com quatro salas em 17 × 17 células. A fortaleza inundada é o **caso difícil**, com dezesseis salas em 41 × 41 células. Em cada caso, UCS e as duas versões do A* partem exatamente do mesmo estado e usam as mesmas regras de custo.

Para avaliar a precisão no estado inicial, comparamos cada estimativa `h(S)` ao custo restante ótimo obtido pela UCS. O erro relativo é `|h(S) − custo ótimo| / custo ótimo`. Também mostramos a faixa de valores da heurística nos estados alcançáveis. Uma heurística não admissível pode subestimar em um estado específico; a violação da admissibilidade exige apenas que ela superestime em **algum** estado.

O tempo abaixo é uma medida de uma execução em segundos e varia entre máquinas; para interpretar o desempenho, considere também os nós visitados e a fronteira.

In [130]:
try:
    for nome, mapa in (("CASO FÁCIL — ala externa, 17 × 17", MAPA_FACIL),
                       ("CASO DIFÍCIL — fortaleza inundada, 41 × 41", MAPA_DIFICIL)):
        configurar_mapa_ativo(mapa)
        print(f"\n{nome}")
        print(f"Início: {POSICAO_INICIAL} | Chave: {POSICAO_CHAVE} | Saída: {POSICAO_SAIDA}")
        solucao = busca_custo_uniforme(ESTADO_INICIAL)
        assert solucao["encontrou_solucao"]
        print(f"\nCaminho ótimo da UCS — custo {solucao['custo_total']}, "
              f"{solucao['tamanho_da_solucao']} movimentos:")
        imprimir_mapa(solucao["caminho_estados"])
finally:
    configurar_mapa_ativo(MAPA_DIFICIL)



CASO FÁCIL — ala externa, 17 × 17
Início: (2, 2) | Chave: (13, 4) | Saída: (13, 13)

Caminho ótimo da UCS — custo 52, 42 movimentos:
Legenda: 🧙 início | 🔑 chave | 🚪 saída | 👣 caminho | 🧱 parede | ▫️ pedra | 🟫 pântano | 🟢 slime

🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱
🧱▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️🧱
🧱▫️🧙👣👣👣▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️🧱
🧱▫️▫️🧱🧱👣▫️▫️🧱▫️▫️🟢🟢🧱▫️▫️🧱
🧱▫️▫️🧱▫️👣👣👣👣👣▫️🟢🟢🧱▫️▫️🧱
🧱▫️▫️▫️▫️▫️▫️▫️🧱👣▫️▫️▫️▫️▫️▫️🧱
🧱▫️▫️▫️▫️▫️▫️▫️🧱👣▫️▫️▫️▫️▫️▫️🧱
🧱▫️▫️▫️▫️▫️▫️▫️🧱👣👣👣👣▫️▫️▫️🧱
🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱👣🧱🧱🧱🧱
🧱🧱🧱▫️▫️▫️▫️🧱🧱▫️▫️▫️👣▫️▫️▫️🧱
🧱▫️🟫🟫🟫🟫🟫▫️🧱▫️▫️▫️👣▫️▫️▫️🧱
🧱▫️🟫🟫▫️🧱🧱▫️🧱▫️👣👣👣🟫🟫▫️🧱
🧱▫️🟫🟫▫️▫️🧱👣👣👣👣🧱🟢🟫🟫▫️🧱
🧱▫️🟫🟫🔑👣👣👣🧱👣👣👣🟢🚪🟫▫️🧱
🧱▫️🟫🟫🟫▫️▫️🧱🧱🟫🟫👣👣👣🟢▫️🧱
🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🟫🟫▫️▫️▫️▫️🧱
🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱

CASO DIFÍCIL — fortaleza inundada, 41 × 41
Início: (2, 2) | Chave: (37, 3) | Saída: (2, 38)

Caminho ótimo da UCS — custo 162, 140 movimentos:
Legenda: 🧙 início | 🔑 chave | 🚪 saída | 👣 caminho | 🧱 parede | ▫️ pedra | 🟫 pântano | 🟢 slime

🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱🧱
🧱▫️▫️▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️▫️▫️🧱▫️▫️▫️▫️▫️▫️▫️▫

In [131]:
from collections import deque

def estados_alcancaveis(estado_inicial):
    """Lista os estados acessíveis para calcular a faixa das heurísticas."""
    pendentes = deque([estado_inicial])
    encontrados = {estado_inicial}
    while pendentes:
        for _, sucessor, _ in gerar_sucessores(pendentes.popleft()):
            if sucessor not in encontrados:
                encontrados.add(sucessor)
                pendentes.append(sucessor)
    return encontrados


def avaliar_caso(nome, mapa):
    configurar_mapa_ativo(mapa)
    resultados = {
        "UCS": busca_custo_uniforme(ESTADO_INICIAL),
        "A* admissível": busca_a_estrela(ESTADO_INICIAL, heuristica_admissivel),
        "A* não admissível": busca_a_estrela(ESTADO_INICIAL, heuristica_nao_admissivel),
    }
    assert all(resultado["encontrou_solucao"] for resultado in resultados.values())
    otimo = resultados["UCS"]["custo_total"]
    assert resultados["A* admissível"]["custo_total"] == otimo

    estados = estados_alcancaveis(ESTADO_INICIAL)
    heuristicas = {
        "A* admissível": heuristica_admissivel,
        "A* não admissível": heuristica_nao_admissivel,
    }
    estimativas = {}
    for nome_heuristica, funcao in heuristicas.items():
        valores = [funcao(estado) for estado in estados]
        inicial = funcao(ESTADO_INICIAL)
        estimativas[nome_heuristica] = {
            "inicial": inicial,
            "faixa": (min(valores), max(valores)),
            "erro_percentual": 100 * abs(inicial - otimo) / otimo,
        }
    return {"nome": nome, "otimo": otimo, "resultados": resultados,
            "estimativas": estimativas}


try:
    comparacao = [
        avaliar_caso("Fácil (ala externa)", MAPA_FACIL),
        avaliar_caso("Difícil (fortaleza)", MAPA_DIFICIL),
    ]
finally:
    configurar_mapa_ativo(MAPA_DIFICIL)

assert comparacao[1]["otimo"] > comparacao[0]["otimo"]
assert (comparacao[1]["resultados"]["UCS"]["total_nos_visitados"]
        > comparacao[0]["resultados"]["UCS"]["total_nos_visitados"])

print("DESEMPENHO — mesmos casos para os três algoritmos")
print(f"{'Caso':<25} {'Método':<19} {'Custo':>5} {'Nós':>5} {'Mov.':>5} {'Tempo (s)':>11} {'Fronteira':>9}")
for caso in comparacao:
    for metodo, resultado in caso["resultados"].items():
        print(f"{caso['nome']:<25} {metodo:<19} {resultado['custo_total']:>5} "
              f"{resultado['total_nos_visitados']:>5} {resultado['tamanho_da_solucao']:>5} "
              f"{resultado['tempo_de_execucao']:>11.6f} "
              f"{resultado['maior_tamanho_da_fronteira']:>9}")

print("\nPRECISÃO DAS HEURÍSTICAS — estado inicial de cada caso")
print(f"{'Caso':<25} {'Heurística':<19} {'Ótimo':>5} {'h(S)':>5} {'Faixa h':>10} {'Erro %':>8}")
for caso in comparacao:
    for nome_heuristica, dados in caso["estimativas"].items():
        faixa = f"{dados['faixa'][0]}–{dados['faixa'][1]}"
        print(f"{caso['nome']:<25} {nome_heuristica:<19} {caso['otimo']:>5} "
              f"{dados['inicial']:>5} {faixa:>10} {dados['erro_percentual']:>7.1f}%")


DESEMPENHO — mesmos casos para os três algoritmos
Caso                      Método              Custo   Nós  Mov.   Tempo (s) Fronteira
Fácil (ala externa)       UCS                    52   243    42    0.000731        15
Fácil (ala externa)       A* admissível          52   197    42    0.000756        30
Fácil (ala externa)       A* não admissível      52   107    42    0.000319        39
Difícil (fortaleza)       UCS                   162  2469   140    0.006728        55
Difícil (fortaleza)       A* admissível         162  1097   140    0.004340       189
Difícil (fortaleza)       A* não admissível     175   349   150    0.001767       155

PRECISÃO DAS HEURÍSTICAS — estado inicial de cada caso
Caso                      Heurística          Ótimo  h(S)    Faixa h   Erro %
Fácil (ala externa)       A* admissível          52    22       0–32    57.7%
Fácil (ala externa)       A* não admissível      52    44       0–64    15.4%
Difícil (fortaleza)       A* admissível         162   106 